# DATA 620 — Week 2 Assignment: Centrality Measures Planning Document

**Course:** DATA 620 – Web Analytics  
**Student:** Zoran Glisovic  
**Date:** 2026-06-15

> **Note on data source:** The original plan used Spotify's Web API endpoint `/artists/{id}/related-artists`. On November 27, 2024, Spotify restricted access to several Web API features, including Related Artists, for new apps and apps still in development mode without extended access. In this project, that endpoint returned HTTP 403, so Last.fm was selected as a practical replacement. Last.fm's `artist.getsimilar` endpoint provides similar artists with match scores through a free API key, making it suitable for constructing an artist-similarity network, although the results come from Last.fm's own data and are not identical to Spotify's related-artist recommendations.

## Dataset: Last.fm Artist Similar-Artist Network

The Last.fm Web API exposes an `artist.getsimilar` endpoint that returns up to 100 artists Last.fm considers similar to a given artist, based on listener behavior and collaborative filtering. Each similar-artist relationship represents an edge in a graph where artists are nodes, weighted by a similarity match score (0–1).

Starting from a manually curated seed list of ~25 well-known artists spanning six genres — pop, hip-hop, R&B, rock, electronic, and Latin — and expanding one level outward via the similar-artists endpoint, we get a network of approximately 150–200 artists with meaningful cross-genre connections.

**API access:** Register at last.fm/api to obtain a free API key. No OAuth or user login required — all requests use a single API key parameter. Accessed via the `requests` library.

```python
import requests

API_KEY = "your_api_key_here"
BASE_URL = "https://ws.audioscrobbler.com/2.0/"

params = {
    "method": "artist.getsimilar",
    "artist": artist_name,
    "api_key": API_KEY,
    "format": "json",
    "limit": 20
}
response = requests.get(BASE_URL, params=params)
similar = response.json()["similarartists"]["artist"]
```

**Categorical variable:** Primary genre, derived from Last.fm tags via the `artist.gettoptags` endpoint, which returns a ranked list of listener-assigned tags per artist (e.g., `['pop', 'dance pop', 'female vocalists']`). We reduce this to a single primary genre using a priority mapping to one of six canonical categories: `pop`, `hip-hop`, `r&b`, `rock`, `electronic`, `latin`.

This satisfies the assignment's categorical variable requirement: each node (artist) has one categorical attribute (genre).

## Data Loading Plan

1. **Obtain API key** from last.fm/api — free, no subscription required.
2. **Define seed artists** — choose approximately 4 artists per genre category (~24 total across pop, hip-hop, R&B, rock, electronic, and Latin), manually selected to represent a range of popularity levels within each genre.
3. **BFS expansion** — for each seed artist, call `artist.getsimilar` (up to 20 results per call). Add each returned artist as a node; add an undirected edge from seed to similar artist weighted by the match score.
4. **Fetch genre tags** — for each unique artist in the network, call `artist.gettoptags` to retrieve their top listener tags. Map the highest-priority matching tag to one of six canonical genre labels using a priority dictionary.
5. **Deduplicate** — the same artist may appear as a similar artist for multiple seeds. Use artist name (normalized to lowercase) as the unique node identifier.
6. **Build the NetworkX graph** — `nx.Graph()` with nodes as artist names, `genre` as a node attribute, and edges representing similar-artist relationships weighted by match score.
7. **Filter** — remove artists where no canonical genre could be assigned (ambiguous tags only), to keep the categorical analysis clean.

Estimated graph size: ~150–200 nodes, ~600–1,000 edges.

**API reference:**
- `artist.getSimilar`: https://www.last.fm/api/show/artist.getSimilar
- `artist.getTopTags`: https://www.last.fm/api/show/artist.getTopTags


## Hypothetical Outcome

**Hypothesis:** Artists in the *pop* genre will show significantly higher average degree centrality than artists in the *rock* genre.

Pop is a genre defined partly by its cross-genre borrowing — pop artists frequently incorporate hip-hop production, R&B vocal styles, and electronic instrumentation. This means pop artists are likely to appear as similar artists across multiple genre clusters, giving them more connections in the network (higher degree). Rock, by contrast, tends toward genre-internal similarity — rock artists are predominantly similar to other rock artists, resulting in a denser intra-genre cluster with fewer cross-genre edges.

**Prediction:** A two-sample t-test on degree centrality scores between the `pop` group and the `rock` group will yield p < 0.05, indicating that pop artists have statistically significantly higher average degree centrality than rock artists.

**Eigenvector centrality** would add a second layer: if pop artists are connected not just to many artists but to *other well-connected artists* (e.g., high-degree hip-hop nodes), their eigenvector scores would exceed what degree centrality alone would predict. This would suggest that pop functions as the genre with the highest network influence — not just the most connections, but the best-placed connections.

## Relevance to Project 1

The dataset, loading plan, and hypothesis above form the basis for Project 1, where the actual API calls, graph construction, centrality calculations, and statistical comparison will be implemented and run. See the related Project 1 notebook, where the actual API calls and analysis will be implemented.